In [106]:
# 08/13/2025
# Trying to fix global issues

using Random, Distributions, Optim, NLsolve, SpecialFunctions

#   ---------  Constants ----------------
N = 2 # Number of countries
periods = 6
J = 3 # Number of goods (needs to be big number and an integer type)
θ = fill(4.0, J) # Frechet shape parameter (governs comparative advantage)

v = 1.0 # migration elasticity
β = .99 # discount factor
α = ones(J) * (1/J) # final good expenditure share
# ----------------------------------------

τ = ones(N, N, J) * 2 # Iceberg trade costs 
for j = 1:J
    for n = 1:N
        τ[n,n,j] = 1
    end
end
Lt = ones(N, J, periods) # Size of labor force in each country at time 
Lt[1,1,1] = 1.0

Ldot = ones(N, J, periods)

At0 = ones(N, J)#initial productivities

wt = ones(N, J, periods)
wt0 = ones(N, J)

wdot = ones(N, J, periods)
tradesharest0_array = ones(N, N, J) * (1 / N)
Adot = ones(N, J, periods)

kdot = ones(N, N, J, periods)
pdotArray = ones(N, J, periods)
d1wdot = ones(N, J)

μtminus1 = zeros(N, N, J, J) #value of μ at time t = -1
πt0 = ones(N, N, J) #initial trade shares
#μt = ones(N, N, J, J, periods)*(1/(N*J))

immobile_share = 0.5 # arbitrary but must be greater than 1/(N*J)
function mu_version1(N, J, periods, immobile_share_xx) # initial migration flows that represent identical moving costs other than staying in countrysector (n,j)
    # α = weight for diagonals (must satisfy 1/(N*J) < α < 1)
    mobile_share = (1 - immobile_share_xx) / (N*J - 1)
    mu = fill(mobile_share, N, N, J, J, periods)
    for i in 1:N, j in 1:J, t in 1:periods
        mu[i, i, j, j, t] = immobile_share_xx
    end
    return mu
end

μt = mu_version1(N, J, periods, immobile_share)



# a guess for path of udot:
udotPathGuess = 1.1*ones(N, J, periods+1)
udotPathUpdate = zeros(N, J, periods+1)
udotPathUpdate[:,:,periods:periods+1] .= 1.0
errormax = 1.0 #the maximum log difference between guesses and updates for appendix D algorithm


function pdot(n, j, d1wdot_xx, kdot, Adot, time, tradesharest0_xx) #pdot(nj) from equation (12)
    (sum(tradesharest0_xx[n, i, j] * (d1wdot_xx[i, j] * kdot[n, i, j, time])^-θ[j] * Adot[i, j, time]^θ[j] for i in 1:N))^(-1 / θ[j])
end 

function tradeSharest0(n, i, j, wt0_xx, At0, τ) #trade shares to nj from ij, equation (7)
    (wt0_xx[i, j] * τ[n, i, j])^(-θ[j]) * At0[i, j] ^θ[j] / (sum((wt0_xx[m, j] *τ[n, m, j])^(-θ[j]) * At0[m, j]^θ[j] for m in 1:N)) 
end

function tradeSharest1(n, i, j, d1wdot_xx, Adot, kdot, time, tradesharest0_xx) #trade shares to nj from ij, equation (13)
    tradesharest0_xx[n, i, j] * ((d1wdot_xx[i, j] * kdot[n, i, j, time]) / pdot(n, j, d1wdot_xx, kdot, Adot, time, tradesharest0_xx))^-θ[j] * Adot[i, j, time]^θ[j]
end

function incomet0(n, j, wt0_xx, Lt_xx) # total income of country n, sector j in time t=0 given wage and labor  
    wt0_xx[n, j] * Lt_xx[n, j, 1] 
end

function incomet1(n, j, d1wdot_xx, time, Lt_xx, wt_xx, Ldot_xx)# total income of country n, sector j in time t+1
    wt_xx[n, j, time]*(d1wdot_xx[n, j])*Lt_xx[n, j, time]*(Ldot_xx[n, j, time])
end

function Xt0(n, j, α, wt0_xx, Lt_xx) # expenditure on sector good j in region n, from equation (8)
    α[j] * sum(wt0_xx[n, k] * Lt_xx[n, k, 1] for k in 1:J) 
end

function Xt1(n, j, α, d1wdot_xx, Lt_xx, wt_xx, Ldot_xx, time) # expenditure on sector good j in region n, from equation (14)
    α[j] * sum(d1wdot_xx[n, k] * Ldot_xx[n, k, time] * wt_xx[n, k, time] * Lt_xx[n, k, time] for k in 1:J) #from equation (14) 
end

function g!(G, wt_xx, Lt_xx, α, At0, τ, θ, N, J)
    # reshape the vector of wages into N×J matrix
    wt0_xx = reshape(wt_xx[1:N*J], N, J)

    idx = 1
    for n in 1:N, j in 1:J
        # total income in country n, sector j
        income_nj = incomet0(n, j, wt0_xx, Lt_xx)
        
        # total expenditure in country n, sector j
        expenditure_nj = sum(Xt0(i, j, α, wt0_xx, Lt_xx) * tradeSharest0(i, n, j, wt0_xx, At0, τ) for i in 1:N)
        
        G[idx] = income_nj - expenditure_nj
        idx += 1
    end
    # normalize first wage
    G[N*J + 1] = wt0_xx[1,1] - 1.0
end

function f!(F, d1wdot_xx, wt_xx, Lt_xx, Ldot_xx, Adot, kdot, tradesharest0_xx, α, θ, N, J, time)
    # reshape input vector into N×J matrix
    d1wdot_xx = reshape(d1wdot_xx[1:N*J], N, J)
    idx = 1
    for n in 1:N, j in 1:J
        income_nj = incomet1(n, j, d1wdot_xx, time, Lt_xx, wt_xx, Ldot_xx)
        expenditure_nj = sum(tradeSharest1(i, n, j, d1wdot_xx, Adot, kdot, time, tradesharest0_xx) * Xt1(i, j, α, d1wdot_xx, Lt_xx, wt_xx, Ldot_xx, time) for i in 1:N)
        F[idx] = income_nj - expenditure_nj
        idx += 1
    end
end

initial = [1.00, 1.00, 1.00, 3.0, 5.0, 3.0, 1.00]   # size: 2×3 if N=2, J=3
results = nlsolve((G, w) -> g!(G, w, Lt, α, At0, τ, θ, N, J), initial)

# extract the wage solution as an N×J matrix
wt0 = reshape(results.zero[1:N*J], N, J)

wt[:,:, 1] .= wt0[:,:] 


println(wt0)  #end solving for wt0

while errormax > .00001
    for n in 1:N
        for i in 1:N
            for j in 1:J
                tradesharest0_array[n, i , j] = tradeSharest0(n, i, j, wt0, At0, τ)
            end
        end
    end

    for time in 1:periods-1

        # Update μt step 2 of appendix D
        for n in 1:N               # destination country
            for i in 1:N           # source country
                for j in 1:J       # destination sector
                    for k in 1:J   # source sector
                        num = μt[n, i, j, k, time] *
                            (udotPathGuess[i, k, time+1])^(β/v)

                        # denominator: sum over all (m, h)
                        denom = sum(μt[n, m, j, h, time] * (udotPathGuess[m, h, time+1])^(β/v) for m in 1:N, h in 1:J)
                        μt[n, i, j, k, time+1] = num / denom
                    end
                end
            end
        end

        # Update Lt step 3 of appendix D
        for n in 1:N
            for j in 1:J
                Lt[n, j, time+1] = sum(μt[i, n, k, j, time] * Lt[i, k, time] for i in 1:N, k in 1:J)
            end
        end
        # Update Ldots from Lts
        Ldot[:, :, time] .= Lt[:, :, time+1] ./ Lt[:, :, time]
    end

    println("Ldot = ", Ldot)

    for time in 1:periods-1
        println("checkpoint")

        initial = [1.0, 2.11, 1.2, 1.15, 5.22, 1.21] 
        res_f = nlsolve((F, x) -> f!(F, x, wt, Lt, Ldot, Adot, kdot, tradesharest0_array, α, θ, N, J, time), initial)
        d1wdot = reshape(res_f.zero[1:N*J], N, J)
        wdot[:, :, time] .= d1wdot[:,:] # updates wdot with solution to the system of equations

        for n in 1:N, j in 1:J
            println(wdot[n, j, time])
        end

        println("checkpoint")

        for n in 1:N, j in 1:J
            pdotArray[n,j,time] = pdot(n, j, d1wdot, kdot, Adot,time, tradesharest0_array)
        end
    
        tradesharest0_array[:,:,:] .= [tradeSharest1(n,i,j,d1wdot,Adot,kdot,time,tradesharest0_array) for n in 1:N, i in 1:N, j in 1:J]

        wt[:,:,time + 1] .= wdot[:, :, time] .* wt[:, :, time]
    end
    println("udotPathGuess = ", udotPathGuess)
    for time in 1:periods-1
        for n in 1:N, j in 1:J
            udotPathUpdate[n,j,time] = wdot[n,j,time]*(sum(μt[n,i,j,k,time]*(udotPathGuess[i,k,time+1])^(β/v) for i in 1:N, k in 1:J))^(v) ##needs verification but is equation 17 as specified by step 5
        end## Note: trying to figure out how to make sure the time+1 does not end the program with infs or NaN in the error
    end
    println("udotPathUpdate = ", udotPathUpdate)
    println("L in periods = ", Lt[:,:,periods-1])

    # Take the log difference of the guess and updated udots to get the error
    #logudotPathGuess = log.(udotPathGuess) 

    #logudotPathUpdate = log.(udotPathUpdate)
    #logdifference = abs.(logudotPathGuess[:,:,1:5] - logudotPathUpdate[:,:,1:5])
    #errormax = maximum(logdifference[:,:,1:5])
    ratiodifference = abs.((udotPathUpdate./udotPathGuess) .- 1)
    errormax = maximum(ratiodifference[:,:,1])

    udotPathGuess = udotPathUpdate
    println("WHILE LOOP")
    display(errormax)
end

[1.0 0.9999999999996319 0.9999999999938249; 1.0000000000045617 1.0000000000049312 1.0000000000107352]
Ldot = [0.9999999999999999 0.9999999999999999 1.0; 0.9999999999999999 1.0 1.0;;; 1.0 1.0 1.0; 1.0 1.0 1.0;;; 1.0 1.0 1.0; 1.0 1.0 1.0;;; 1.0 1.0 1.0; 1.0 1.0 1.0;;; 1.0 1.0 1.0; 1.0 1.0 1.0;;; 1.0 1.0 1.0; 1.0 1.0 1.0]
checkpoint
-3.4972506991664156e-5
-3.497267544709136e-5
-3.49716689793457e-5
-3.497563522488309e-5
-3.4975466769455786e-5
-3.497647323720147e-5
checkpoint
checkpoint
0.14848078483441393
0.14849346608145494
0.14841722654157571
0.14858219197270767
0.14856951222610335
0.14864573910576667
checkpoint
checkpoint
3.168977765600223
3.168691875300067
3.170410797251244
3.166531724851957
3.1668172254725957
3.1651021970096815
checkpoint
checkpoint
-0.04772188531103695
-0.04772618044990063
-0.04770068214344268
-0.047758430959049
-0.04775413582024042
-0.04777963412594646
checkpoint
checkpoint
-2.6619293113381923
-2.6616897465927623
-2.6631133593053535
-2.65989187040211
-2.660131111650

1.000034943152979

DomainError: DomainError with -0.05244406548129006:
Exponentiation yielding a complex result requires a complex argument.
Replace x^y with (x+0im)^y, Complex(x)^y, or similar.